# Visual Archive Activator — SD 1.5 + ControlNet
Developed by Federico Fort. Companion tool to the Sage Research Methods dataset *Generative AI as Archival Intervention: Reimagining Bolivian Photographic Archives (1925–1935)*.

1. Choose a GPU: **Runtime → Change runtime type → T4 GPU**.
2. Run the cells in order. The last cell downloads the app (version `v1.0`) from GitHub and prints a public Gradio link: open it to use the interface.
3. The first generation downloads the models (about 4 GB) and takes several minutes.

In [ ]:
!nvidia-smi
import sys
print('Python:', sys.version)

In [ ]:
# Installs the app's dependencies quietly. pip's full output is saved to pip_install.log.
import importlib.util, subprocess, sys

def pip_install(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       capture_output=True, text=True)
    log = r.stdout + r.stderr
    with open("pip_install.log", "a") as f:
        f.write(log)
    if r.returncode != 0:
        print(log)
        raise RuntimeError("Installation failed. See the messages above.")
    return log

# Colab already ships a CUDA build of PyTorch; install one only if it's missing (e.g. a fresh local runtime).
if importlib.util.find_spec("torch") is None:
    pip_install("torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu128")

log = pip_install(
    "diffusers==0.35.1", "transformers==4.54.1", "accelerate==1.10.1", "safetensors==0.6.2",
    "gradio==5.44.1", "controlnet-aux==0.0.10", "opencv-python==4.11.0.86", "Pillow==11.3.0",
    "numpy==2.2.6", "huggingface-hub==0.34.4", "invisible-watermark==0.2.0",
)

import torch
assert torch.cuda.is_available(), "No active GPU. Choose T4 GPU and restart the runtime."
print("✓ Dependencies installed.")
print("  PyTorch", torch.__version__, "on", torch.cuda.get_device_name(0))
if "dependency conflicts" in log:
    print("  Note: pip reported version conflicts with packages that come preinstalled on Colab")
    print("  (e.g. google-genai, google-adk, hf-gradio). This app doesn't use them, so it's safe")
    print("  to continue. Full details are in pip_install.log.")

In [ ]:
# If the base model requires authorization, uncomment these two lines:
# from huggingface_hub import login
# login()

import os
from pathlib import Path

REPO_URL = "https://github.com/federicofif-cyber/visual-archive-activator.git"
VERSION = "v1.0"  # the release that matches the Sage dataset; change it to use another version

# On a fresh Colab VM, fetch the app from GitHub at the pinned version.
# If app.py is already next to this notebook (e.g. a local runtime pointed at
# the project folder), that copy is used instead and nothing is downloaded.
if not Path("app.py").exists():
    if not Path("visual-archive-activator/app.py").exists():
        !git clone --quiet --depth 1 --branch {VERSION} {REPO_URL}
    os.chdir("visual-archive-activator")

!python app.py